In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [11]:
import importlib
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from compare_models import (
    build_features,
    choose_features,
    fit_dr_xlearner,
    fit_plain_xlearner,
    fit_plain_catboost,
    evaluate_all_models,
    fit_convex_ensemble_simplex,
    fit_ridge_ensemble,
    apply_ensemble_to_comparison,
    print_ensemble,
)

# importlib.reload(cm)  # safe if you've just edited the file

In [7]:
# 1) Load data
train = pd.read_csv("../train_data.csv")
test = pd.read_csv(
    "../test_data.csv"
)  # must contain the target if you want actual iROAS

# 2) Feature pipeline
train_f = build_features(train)
test_f = build_features(test)
feats, cats = choose_features(train_f)

## Insights:

- Blend successfully cancels biases: DR_X/X underestimate, Plain overestimates, Blend centers well
- Generalization gap is acceptable: Test MAE ~2.5x training MAE suggests mild overfitting but still strong performance
- DR_X instability on test: Performance degradation suggests it may be overfitting to training campaign patterns
- Business impact: Blend predictions are within 12% of actual iROAS, sufficient for campaign targeting decisions

## Train Inidividual Models

In [ ]:
# 3) Fit models (log-target + smearing inside each fit)
m_dr = fit_dr_xlearner(train_f, feats, cats, cap_q=0.995, use_propensity=True, seed=42)
m_x = fit_plain_xlearner(train_f, feats, cats, seed=42)
m_cb = fit_plain_catboost(train_f, feats, cats, include_treatment=True, seed=42)

print(
    "Plain baseline -> log_target:", m_cb.get("log_target"), "smear:", m_cb.get("smear")
)

# 4) Compare on TRAIN (cap actuals to DR cap for apples-to-apples)
comp_train = evaluate_all_models(
    train_f,
    {"DR_X": m_dr, "X": m_x, "Plain": m_cb},
    feats,
    cats,
    cap_actual_at=m_dr["cap_value"],
)

# 5) Compare on TEST (same cap alignment)
#    If test doesn't have the target column, skip evaluate_all_models and just do predictions:
#    preds_test_plain = cm.summarize_campaigns(test_f, m_cb)  # predicted iROAS only
comp_test = cm.evaluate_all_models(
    test_f,
    {"DR_X": m_dr, "X": m_x, "Plain": m_cb},
    feats,
    cats,
    cap_actual_at=m_dr["cap_value"],
)

Plain baseline -> log_target: True smear: 6.288758193045096
TRAIN (iROAS cols only):


,iROAS_actual,iROAS_pred_DR_X,iROAS_pred_X,iROAS_pred_Plain
0,"1,615.12","1,351.73","1,272.63","2,106.87"
1,"1,632.56","1,337.45","1,263.70","2,185.00"
2,"2,180.34","1,732.43","1,661.55","2,662.09"


TEST (iROAS cols only):


,iROAS_actual,iROAS_pred_DR_X,iROAS_pred_X,iROAS_pred_Plain
0,"1,947.24","1,301.78","1,535.71","2,468.85"
1,"1,836.75","1,160.32","1,268.73","2,070.36"
2,"1,579.38","1,265.57","1,378.35","2,086.46"


Wrote: comparison_train.csv, comparison_test.csv, train_iROAS_scatter_DR_X.png, test_iROAS_scatter_DR_X.png


## Train Blend Model

In [9]:
# 6) Define the blend model
ens_cvx = fit_convex_ensemble_simplex(
    comp_train, model_keys=["DR_X", "X", "Plain"], step=0.01
)
print_ensemble(ens_cvx)

# 7) Apply to TRAIN and TEST
comp_train_ens = apply_ensemble_to_comparison(comp_train, ens_cvx, out_name="Blend")
comp_test_ens = apply_ensemble_to_comparison(comp_test, ens_cvx, out_name="Blend")

# 8) Inspect iROAS columns (you like this view)
display(comp_train_ens.filter(regex="^iROAS"))
display(comp_test_ens.filter(regex="^iROAS"))

Ensemble -> DR_X:0.000 + X:0.570 + Plain:0.430 + intercept:0.000


,iROAS_actual,iROAS_pred_DR_X,iROAS_pred_X,iROAS_pred_Plain,iROAS_pred_Blend
0,"1,615.12","1,351.73","1,272.63","2,106.87","1,631.36"
1,"1,632.56","1,337.45","1,263.70","2,185.00","1,659.86"
2,"2,180.34","1,732.43","1,661.55","2,662.09","2,091.78"


,iROAS_actual,iROAS_pred_DR_X,iROAS_pred_X,iROAS_pred_Plain,iROAS_pred_Blend
0,"1,947.24","1,301.78","1,535.71","2,468.85","1,936.96"
1,"1,836.75","1,160.32","1,268.73","2,070.36","1,613.43"
2,"1,579.38","1,265.57","1,378.35","2,086.46","1,682.84"


In [14]:
def error_summary(comp, keys=("DR_X", "X", "Plain", "Blend")):
    rows = []
    y = comp["iROAS_actual"].values.astype(float)
    for k in [k for k in keys if f"iROAS_pred_{k}" in comp.columns]:
        yhat = comp[f"iROAS_pred_{k}"].values.astype(float)
        mae = float(np.mean(np.abs(yhat - y)))
        rmse = float(np.sqrt(np.mean((yhat - y) ** 2)))
        rows.append({"model": k, "MAE": mae, "RMSE": rmse})
    return pd.DataFrame(rows).sort_values("MAE")


display(error_summary(comp_train_ens))
display(error_summary(comp_test_ens))

,model,MAE,RMSE
3,Blend,44.03,54.32
0,DR_X,335.47,345.01
1,X,410.04,417.33
2,Plain,508.65,509.61


,model,MAE,RMSE
3,Blend,112.35,142.22
1,X,393.53,421.28
2,Plain,420.77,441.13
0,DR_X,545.24,569.40


## Save Blend Model

In [ ]:
from blended_ensemble_artifact import BlendedTauDF
import joblib


def _extract_weights(ens, order=("DR_X", "X", "Plain")):
    """
    Returns (weights_map: dict[str,float], intercept: float).
    Accepts:
      - {"weights": {...}, "intercept": ...}
      - {"DR_X":..., "X":..., "Plain":..., "intercept": ...}
      - list/tuple/np.ndarray aligned to `order`
      - object with .weights or attributes
    """
    if isinstance(ens, dict):
        if "weights" in ens:
            w = ens["weights"]
            b = float(ens.get("intercept", 0.0))
            if isinstance(w, (list, tuple, np.ndarray)):
                w = np.asarray(w, dtype=float)
                if w.shape[0] != len(order):
                    raise ValueError(
                        "Weights array length does not match expected order."
                    )
                wmap = {k: float(w[i]) for i, k in enumerate(order)}
            else:
                wmap = {k: float(w.get(k, 0.0)) for k in order}
            return wmap, b
        else:
            b = float(ens.get("intercept", 0.0))
            wmap = {k: float(ens.get(k, 0.0)) for k in order}
            return wmap, b

    if isinstance(ens, (list, tuple, np.ndarray)):
        arr = np.asarray(ens, dtype=float)
        if arr.shape[0] != len(order):
            raise ValueError("Weights array length does not match expected order.")
        wmap = {k: float(arr[i]) for i, k in enumerate(order)}
        return wmap, 0.0

    if hasattr(ens, "weights"):
        return _extract_weights(getattr(ens, "weights"), order)

    # object with named attributes
    wmap = {k: float(getattr(ens, k)) for k in order}
    b = float(getattr(ens, "intercept", 0.0))
    return wmap, b


# 2) Pull weights & intercept from your ens_cvx
wmap, b = _extract_weights(ens_cvx, order=("DR_X", "X", "Plain"))
print("weights:", wmap, "intercept:", b)

# 3) Build the single blended learner (NOTE: intercept=b, not `bias`)
blended = BlendedTauDF(m_dr=m_dr, m_x=m_x, m_plain=m_cb, weights=wmap, intercept=b)

# 4) Save artifacts
ART_DIR = "../artifacts"
os.makedirs(ART_DIR, exist_ok=True)

joblib.dump(blended, os.path.join(ART_DIR, "learner.joblib"))

cap_val_safe = float(cap_val) if "cap_val" in globals() else float("nan")
info = {
    "learner_kind": "df_blend",
    "weights": {k: float(v) for k, v in wmap.items()},
    "intercept": float(b),
    "cap_val": cap_val_safe,
    "uses_campaign_id_feature": False,
}
joblib.dump(info, os.path.join(ART_DIR, "info.joblib"))

print("Saved blended learner to", os.path.join(ART_DIR, "learner.joblib"))